## Basic bigram modeling

In [ ]:
with open("names.txt") as f:
    words = f.read().splitlines()

In [ ]:
b = {}
for w in words:
    chs = ['<S>'] + list(w) + ['<E>']
    for ch1, ch2 in zip(chs, chs[1:]):
        bigram = (ch1, ch2)
        b[bigram] = b.get(bigram, 0) + 1

In [ ]:
import torch

N = torch.zeros((27, 27), dtype=torch.int32)

chars = sorted(list(set(''.join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i: s for s, i in stoi.items()}

In [ ]:
for word in words:
    chs = ['.'] + list(word) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

plt.figure(figsize=(16, 16))
plt.imshow(N, cmap='Greens')
for i in range(27):
    for j in range(27):
        chstr = itos[i] + itos[j]
        plt.text(j, i, str(N[i, j].item()), ha="center", va="top", color='black')
        plt.text(j, i, chstr, ha="center", va="bottom", color='black')
plt.axis('off');

In [ ]:
P = N.float()
P /= P.sum(1, keepdims=True)
g = torch.Generator().manual_seed(1)

generated_words = []
for i in range(10):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    w = ''.join(out[:-1])
    generated_words.append(w)
    print(w)

In [ ]:
def compute_avg_nll(words):
    log_likelihood = 0.0
    n = 0

    for w in words:
        chs = ['.'] + list(w) + ['.']
        for ch1, ch2 in zip(chs, chs[1:]):
            ix1 = stoi[ch1]
            ix2 = stoi[ch2]
            prob = P[ix1, ix2]
            logprob = torch.log(prob)
            log_likelihood += logprob
            n += 1

    nll = -log_likelihood
    avg_nll = nll / n
    return avg_nll.item()

In [ ]:
print("\nTraining data loss:")
print(compute_avg_nll(words))

print("\nInference data loss:")
print(compute_avg_nll(generated_words))

## Bigram modeling - in the neural network framework

### Create the training dataset

In [ ]:
# create the training set of bigrams (x,y)
xs, ys = [], []

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        xs.append(ix1)
        ys.append(ix2)

xs = torch.tensor(xs)
ys = torch.tensor(ys)
num = xs.nelement()
print('number of samples: ', num)

### Initialize NN

In [ ]:
# The following line represents the first layer of the NN. Each row of W is a "neuron", with each column representing
# a weight dimension for that neuron. So, this layer of the NN has 27 neurons each with 27 weights.
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((27, 27), generator=g, requires_grad=True)

### Gradient descent (i.e., the "training")

In [ ]:
import torch.nn.functional as F

# training hyperparameters
EPOCHS = 10
LEARNING_RATE = 50

# gradient descent
for k in range(EPOCHS):
    # forward pass
    xenc = F.one_hot(xs, num_classes=27).float()  # input to the network: one-hot encoding
    logits = xenc @ W  # predict log-counts
    counts = logits.exp()  # counts, equivalent to N
    probs = counts / counts.sum(1, keepdims=True)  # probabilities for next character
    # advanced indexing in PyTorch - essentially, what "probs[torch.arange(num), ys]" does it pick the
    # prob assigned by the model for the expected next character (as in "ys") for each of the given input
    # character, log, average and negate them to form the NLL value
    loss = -probs[torch.arange(num), ys].log().mean()
    print(loss.item())

    # backward pass
    W.grad = None  # set to zero the gradient
    loss.backward()

    # update
    W.data += -LEARNING_RATE * W.grad


### Inference

In [ ]:
# finally, sample from the 'neural net' model
g = torch.Generator().manual_seed(1)

for i in range(10):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()
        logits = xenc @ W  # predict log-counts
        counts = logits.exp()  # counts, equivalent to N
        p = counts / counts.sum(1, keepdims=True)  # probabilities for next character
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:
            break
    print(''.join(out))